# SAMATRIX RESUMEFORGE 2026: PHASE 3
## TEXT PREPROCESSING & FEATURE ENGINEERING
**Stages Covered: Stage 5 (Text Preprocessing - 10 Marks), Stage 6 (Train/Val/Test Split - 5 Marks), Stage 7 (Feature Engineering - 5 Marks)**
**Total Marks for Phase 3: 20 / 70 Marks**

---

### 📋 Common Context Prompt
> **Project**: ResumeForge 2026 — End-to-End NLP Resume Classification System  
> **Problem Statement**: Build an automated NLP system that takes unstructured resume text and classifies it into its appropriate industry sector among 24 categories.  
> **Workflow**: Problem → Data Gathering → Quality Check & Leakage Prevention → EDA → Preprocessing → Features → ML/DL → Evaluation → Demo  
> **Phase 3 Objectives**: Implement a reusable resume-cleaning function for training and inference; eliminate HTML, emails, phone numbers, URLs, and formatting noise while safeguarding technical tokens (`Python`, `C++`, `C#`, `.NET`, `SQL`, `AWS`, `Docker`, `CI/CD`, `Power BI`, `Java`, `React`, ML terms); execute stratified 70/15/15 train/val/test splitting; fit all learned transformations strictly on training data; build a sublinear TF-IDF (1,2) n-gram pipeline (~12,000 features); document vocabulary size and leakage checks; and persist all artifacts for downstream modeling.  
> **Phase 3 Rule**: Do NOT evaluate on the test set in this phase.


---
## 🛠️ Environment Setup & Library Imports
Initialize data handling, regular expressions, scikit-learn feature extractors, and persistence libraries.


In [1]:
import os
import re
import json
import joblib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')

# Directory paths
DATASET_PATH = Path("../Dataset/Resume/Resume.csv")
MODELS_DIR = Path("../artifacts/models")
REPORTS_DIR = Path("../artifacts/reports")
CACHE_DIR = Path("../artifacts")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Load validated dataset
df = pd.read_csv(DATASET_PATH)
df['Resume_str'] = df['Resume_str'].astype(str)
df['Category'] = df['Category'].astype(str).str.strip().str.upper()

print(f"Validated dataset loaded: {df.shape[0]} resumes, {df['Category'].nunique()} categories.")


Validated dataset loaded: 2484 resumes, 24 categories.


---
## 🧹 Task 1, 2 & 3: Reusable Resume Cleaning Pipeline

A critical requirement from the hackathon guidelines:
> *"Apply preprocessing deliberately. Do not remove information blindly; resume text contains useful skills, technologies, certifications, education terms, job titles, and experience indicators. Preserve meaningful technical tokens such as Python, C++, SQL, AWS, TensorFlow, Java, .NET, NLP, etc."*

Standard regex tokenizers (`\w+`) strip `+`, `#`, `.`, and `/`, destroying terms like `C++`, `C#`, `.NET`, `Node.js`, and `CI/CD`.  
We implement a mapping strategy that transforms multi-character technical tokens into unified protective representations before noise stripping.


In [2]:
# Technical token dictionary: Protect multi-character symbols
TECH_TOKEN_REPLACEMENTS = {
    # Programming languages & frameworks
    r"\bc\+\+(?!\w)": "cplusplus",
    r"\bc#(?!\w)": "csharp",
    r"(?<!\w)\.net(?!\w)": "dotnet",
    r"\bnode\.js\b": "nodejs",
    r"\bvue\.js\b": "vuejs",
    r"\breact\.js\b": "reactjs",
    r"\bangular\.js\b": "angularjs",
    r"\bspring\s+boot\b": "springboot",
    
    # Cloud, DevOps & Tools
    r"\bci\/cd\b": "cicd",
    r"\bpower\s+bi\b": "powerbi",
    r"\bms\s+excel\b": "msexcel",
    r"\bms\s+office\b": "msoffice",
    r"\bms\s+word\b": "msword",
    r"\bms\s+access\b": "msaccess",
    
    # Machine Learning & AI
    r"\bmachine\s+learning\b": "machinelearning",
    r"\bdeep\s+learning\b": "deeplearning",
    r"\bdata\s+science\b": "datascience",
    r"\bcloud\s+computing\b": "cloudcomputing",
    r"\bnatural\s+language\s+processing\b": "nlp",
    r"\blarge\s+language\s+models?\b": "llm",
    
    # Professional Domains
    r"\bpublic\s+relations\b": "publicrelations",
    r"\bbusiness\s+development\b": "businessdevelopment",
    r"\bhuman\s+resources\b": "humanresources",
    r"\bquality\s+assurance\b": "qualityassurance"
}

# Domain-safe stopword list (stopwords removed without pruning technical tokens)
DOMAIN_SAFE_STOPWORDS = {
    "a", "about", "above", "after", "again", "against", "all", "am", "an", "and",
    "any", "are", "aren't", "as", "at", "be", "because", "been", "before", "being",
    "below", "between", "both", "but", "by", "can't", "cannot", "could", "couldn't",
    "did", "didn't", "do", "does", "doesn't", "doing", "don't", "down", "during",
    "each", "few", "for", "from", "further", "had", "hadn't", "has", "hasn't",
    "have", "haven't", "having", "he", "he'd", "he'll", "he's", "her", "here",
    "here's", "hers", "herself", "him", "himself", "his", "how", "how's", "i",
    "i'd", "i'll", "i'm", "i've", "if", "in", "into", "is", "isn't", "it", "it's",
    "its", "itself", "let's", "me", "more", "most", "mustn't", "my", "myself",
    "no", "nor", "not", "of", "off", "on", "once", "only", "or", "other", "ought",
    "our", "ours", "ourselves", "out", "over", "own", "same", "shan't", "she",
    "she'd", "she'll", "she's", "should", "shouldn't", "so", "some", "such",
    "than", "that", "that's", "the", "their", "theirs", "them", "themselves",
    "then", "there", "there's", "these", "they", "they'd", "they'll", "they're",
    "they've", "this", "those", "through", "to", "too", "under", "until", "up",
    "very", "was", "wasn't", "we", "we'd", "we'll", "we're", "we've", "were",
    "weren't", "what", "what's", "when", "when's", "where", "where's", "which",
    "while", "who", "who's", "whom", "why", "why's", "with", "won't", "would",
    "wouldn't", "you", "you'd", "you'll", "you're", "you've", "your", "yours",
    "yourself", "yourselves", "state", "city", "company", "name"
}

def clean_resume_text(text: str, remove_stopwords: bool = True) -> str:
    """
    Reusable, production-grade text cleaning function for training and API inference.
    
    Pipeline Steps:
    1. Case normalization (lowercase).
    2. Protection of technical terms and symbols (C++, C#, .NET, CI/CD, Power BI, ML).
    3. Removal of HTML tags and entities.
    4. Removal of email addresses, URLs, and phone numbers.
    5. Removal of punctuation, special symbols, and isolated digits.
    6. Domain-safe stopword removal (keeps terms >= 2 chars).
    7. Whitespace normalization.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
        
    # 1. Lowercase
    cleaned = text.lower()
    
    # 2. Protect multi-token phrases and technical symbols
    for pattern, replacement in TECH_TOKEN_REPLACEMENTS.items():
        cleaned = re.sub(pattern, f" {replacement} ", cleaned)
        
    # 3. Strip HTML markup & XML artifacts
    cleaned = re.sub(r"<[^>]+>", " ", cleaned)
    cleaned = re.sub(r"&[a-z]+;", " ", cleaned)
    
    # 4. Remove URLs, emails, phone numbers
    cleaned = re.sub(r"http\S+|www\.\S+", " ", cleaned)
    cleaned = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b", " ", cleaned)
    cleaned = re.sub(r"\+?\d[\d\s\-\(\)]{7,}\d", " ", cleaned)
    
    # 5. Remove remaining non-alphanumeric symbols and standalone numbers
    cleaned = re.sub(r"[^a-zA-Z\s]", " ", cleaned)
    
    # 6. Stopword filtering & length pruning
    words = cleaned.split()
    if remove_stopwords:
        words = [w for w in words if w not in DOMAIN_SAFE_STOPWORDS and len(w) >= 2]
    else:
        words = [w for w in words if len(w) >= 2]
        
    # 7. Normalize repeated whitespace
    return " ".join(words)


In [3]:
# 1.2 Verification of Technical Token Preservation on Sample Resume
sample_raw = """
Candidate Profile: Senior Full-Stack Cloud Architect & ML Engineer
Contact: dev.lead@cloudcorp.io | +1 (555) 234-5678 | https://github.com/clouddev
Summary:
Expert in Python, C++, C#, and .NET core microservices architecture.
Extensive experience deploying Docker containers on AWS (EC2, S3, Lambda) with automated CI/CD pipelines.
Designed enterprise reporting dashboards using Power BI and advanced SQL database optimization.
Hands-on with React, Java Spring Boot, and Machine Learning models for predictive analytics.
"""

sample_cleaned = clean_resume_text(sample_raw)
print("="*75)
print("PREPROCESSING VERIFICATION ON TECHNICAL SNIPPET")
print("="*75)
print("Raw Input Snippet:")
print(sample_raw.strip())
print("\nCleaned Output Snippet:")
print(sample_cleaned)
print("="*75)

# Assertions to ensure technical tokens are preserved
assert "cplusplus" in sample_cleaned, "Failed to preserve C++!"
assert "csharp" in sample_cleaned, "Failed to preserve C#!"
assert "dotnet" in sample_cleaned, "Failed to preserve .NET!"
assert "cicd" in sample_cleaned, "Failed to preserve CI/CD!"
assert "powerbi" in sample_cleaned, "Failed to preserve Power BI!"
assert "machinelearning" in sample_cleaned, "Failed to preserve Machine Learning!"
assert "python" in sample_cleaned, "Failed to preserve Python!"
assert "docker" in sample_cleaned, "Failed to preserve Docker!"
assert "aws" in sample_cleaned, "Failed to preserve AWS!"
assert "sql" in sample_cleaned, "Failed to preserve SQL!"
assert "react" in sample_cleaned, "Failed to preserve React!"
assert "java" in sample_cleaned, "Failed to preserve Java!"
assert "dev.lead@cloudcorp.io" not in sample_cleaned, "Failed to strip email!"
assert "http" not in sample_cleaned, "Failed to strip URL!"
print("All technical token preservation assertions PASSED successfully!")


PREPROCESSING VERIFICATION ON TECHNICAL SNIPPET
Raw Input Snippet:
Candidate Profile: Senior Full-Stack Cloud Architect & ML Engineer
Contact: dev.lead@cloudcorp.io | +1 (555) 234-5678 | https://github.com/clouddev
Summary:
Expert in Python, C++, C#, and .NET core microservices architecture.
Extensive experience deploying Docker containers on AWS (EC2, S3, Lambda) with automated CI/CD pipelines.
Designed enterprise reporting dashboards using Power BI and advanced SQL database optimization.
Hands-on with React, Java Spring Boot, and Machine Learning models for predictive analytics.

Cleaned Output Snippet:
candidate profile senior full stack cloud architect ml engineer contact summary expert python cplusplus csharp dotnet core microservices architecture extensive experience deploying docker containers aws ec lambda automated cicd pipelines designed enterprise reporting dashboards using powerbi advanced sql database optimization hands react java springboot machinelearning models predicti

### Apply Preprocessing to Dataset
Now we clean all 2,484 resumes and encode target labels.


In [4]:
# Apply cleaning to full dataset
print("Applying cleaning function to all 2,484 resumes in dataset...")
df['Clean_text'] = df['Resume_str'].apply(clean_resume_text)

# Label Encoding for 24 Categories
label_encoder = LabelEncoder()
df['Category_Encoded'] = label_encoder.fit_transform(df['Category'])

# Save LabelEncoder
label_encoder_path = MODELS_DIR / "label_encoder.joblib"
joblib.dump(label_encoder, label_encoder_path)
print(f"Saved LabelEncoder artifact to {label_encoder_path.resolve()}")
print(f"Total encoded classes: {len(label_encoder.classes_)}")


Applying cleaning function to all 2,484 resumes in dataset...
Saved LabelEncoder artifact to D:\ResumeForge\artifacts\models\label_encoder.joblib
Total encoded classes: 24


---
## 🛡️ Task 4 & 5: Stratified 70/15/15 Splitting & Zero-Leakage Protocol

To ensure strict zero data leakage:
1. **Partition First**: Split the dataset into **70% Train**, **15% Validation**, and **15% Test** before vectorization.
2. **Stratification**: Maintain identical class proportions across all three partitions using `random_state=42`.
3. **Strict Fitting Rule**: The `TfidfVectorizer` will be fitted **exclusively on the training split** (`df_train['Clean_text']`). Validation and test splits will be transformed using the fitted vocabulary.


In [5]:
# 4.1 Execute Stratified 70/15/15 Split
# First split off 30% for Validation + Test
df_train, df_temp = train_test_split(
    df,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df['Category_Encoded'],
    random_state=RANDOM_STATE
)

# Split 30% temp into 15% Val and 15% Test (50/50 split)
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    stratify=df_temp['Category_Encoded'],
    random_state=RANDOM_STATE
)

print("="*65)
print("DATASET STRATIFIED PARTITION BREAKDOWN")
print("="*65)
print(f"• Training Set   : {len(df_train):4d} resumes ({len(df_train)/len(df)*100:.1f}%)")
print(f"• Validation Set : {len(df_val):4d} resumes ({len(df_val)/len(df)*100:.1f}%)")
print(f"• Test Set       : {len(df_test):4d} resumes ({len(df_test)/len(df)*100:.1f}%)")
print(f"• Total Resumes  : {len(df_train) + len(df_val) + len(df_test):4d} resumes")
print("="*65)

# Verify zero missing classes across all splits
train_classes = set(df_train['Category_Encoded'])
val_classes = set(df_val['Category_Encoded'])
test_classes = set(df_test['Category_Encoded'])

assert len(train_classes) == 24, "Missing class in Train set!"
assert len(val_classes) == 24, "Missing class in Validation set!"
assert len(test_classes) == 24, "Missing class in Test set!"
print("Stratification verification PASSED: All 24 classes present in each split.")


DATASET STRATIFIED PARTITION BREAKDOWN
• Training Set   : 1738 resumes (70.0%)
• Validation Set :  373 resumes (15.0%)
• Test Set       :  373 resumes (15.0%)
• Total Resumes  : 2484 resumes
Stratification verification PASSED: All 24 classes present in each split.


---
## ⚙️ Task 6 & 7: Build TF-IDF Feature Pipeline & Leakage-Prevention Checks

We construct the feature extraction pipeline strictly adhering to hackathon specifications:
- `sublinear_tf=True`: Replaces $tf$ with $1 + \log(tf)$ to prevent highly repetitive document terms from dominating feature weights.
- `ngram_range=(1, 2)`: Captures both unigrams (*python*, *accounting*) and informative bigrams (*machine learning*, *tax audit*, *cloud architecture*).
- `max_features=12000`: Caps vocabulary size to the top 12,000 most informative n-grams across the corpus.
- `min_df=2`: Ignores idiosyncratic single-occurrence typos.
- `max_df=0.95`: Filters ubiquitous corpus noise.


In [6]:
# 6.1 Initialize and Fit Vectorizer on Training Data ONLY
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=12000,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode'
)

print("Fitting TfidfVectorizer strictly on Training Set (1,738 resumes)...")
X_train = vectorizer.fit_transform(df_train['Clean_text'])

# Transform Validation and Test sets without refitting
print("Transforming Validation Set (373 resumes) with fitted vectorizer...")
X_val = vectorizer.transform(df_val['Clean_text'])

print("Transforming Test Set (373 resumes) with fitted vectorizer...")
X_test = vectorizer.transform(df_test['Clean_text'])

y_train = df_train['Category_Encoded'].values
y_val = df_val['Category_Encoded'].values
y_test = df_test['Category_Encoded'].values

# Feature metrics
vocab_size = len(vectorizer.vocabulary_)
print("\n" + "="*65)
print("TF-IDF FEATURE MATRIX SUMMARY")
print("="*65)
print(f"• Fitted Vocabulary Size     : {vocab_size:,} features")
print(f"• Training Matrix Shape      : {X_train.shape[0]} rows x {X_train.shape[1]} columns")
print(f"• Validation Matrix Shape    : {X_val.shape[0]} rows x {X_val.shape[1]} columns")
print(f"• Test Matrix Shape          : {X_test.shape[0]} rows x {X_test.shape[1]} columns")
print(f"• Training Matrix Sparsity   : {(1.0 - X_train.nnz / (X_train.shape[0] * X_train.shape[1])) * 100:.2f}% sparse")
print("="*65)


Fitting TfidfVectorizer strictly on Training Set (1,738 resumes)...
Transforming Validation Set (373 resumes) with fitted vectorizer...
Transforming Test Set (373 resumes) with fitted vectorizer...

TF-IDF FEATURE MATRIX SUMMARY
• Fitted Vocabulary Size     : 12,000 features
• Training Matrix Shape      : 1738 rows x 12000 columns
• Validation Matrix Shape    : 373 rows x 12000 columns
• Test Matrix Shape          : 373 rows x 12000 columns
• Training Matrix Sparsity   : 97.00% sparse


In [7]:
# 7.1 Leakage-Prevention Check
# Test that vectorizer features originate solely from the training split
feature_names = np.array(vectorizer.get_feature_names_out())
print("7.1 Leakage-Prevention Verification:")
print(f"• Vectorizer vocabulary fitted strictly on training data: YES ({len(feature_names)} features)")
print(f"• Feature names sample (first 10) : {feature_names[:10].tolist()}")
print(f"• Technical tokens in vocabulary :")
sample_tech_tokens = ["cplusplus", "csharp", "dotnet", "cicd", "powerbi", "machinelearning", "python", "docker", "aws", "sql", "reactjs", "java"]
for token in sample_tech_tokens:
    in_vocab = token in vectorizer.vocabulary_
    print(f"   - {token:18s}: {'Present (Index: ' + str(vectorizer.vocabulary_.get(token)) + ')' if in_vocab else 'Not in top 12k'}")

# Top TF-IDF features across categories in training data
print("\nTop 15 Most Informative Terms in Training Corpus by Mean TF-IDF:")
mean_tfidf = np.asarray(X_train.mean(axis=0)).ravel()
top_feat_indices = mean_tfidf.argsort()[::-1][:15]
df_top_terms = pd.DataFrame({
    'Feature': feature_names[top_feat_indices],
    'Mean TF-IDF': mean_tfidf[top_feat_indices].round(4)
})
display(df_top_terms)


7.1 Leakage-Prevention Verification:
• Vectorizer vocabulary fitted strictly on training data: YES (12000 features)
• Feature names sample (first 10) : ['aa', 'aaa', 'aas', 'ab', 'abc', 'abilities', 'ability', 'ability analyze', 'ability build', 'ability communicate']
• Technical tokens in vocabulary :
   - cplusplus         : Present (Index: 2476)
   - csharp            : Present (Index: 2566)
   - dotnet            : Present (Index: 3267)
   - cicd              : Not in top 12k
   - powerbi           : Not in top 12k
   - machinelearning   : Present (Index: 6072)
   - python            : Present (Index: 8631)
   - docker            : Not in top 12k
   - aws               : Present (Index: 941)
   - sql               : Present (Index: 10253)
   - reactjs           : Not in top 12k
   - java              : Present (Index: 5583)

Top 15 Most Informative Terms in Training Corpus by Mean TF-IDF:


,Feature,Mean TF-IDF
0,management,0.0242
1,sales,0.0216
2,customer,0.0213
3,business,0.0196
4,service,0.0193
5,new,0.0190
6,team,0.0185
7,training,0.0179
8,manager,0.0172
9,project,0.0169


---
## 💾 Task 8: Save Reusable Preprocessing Code & Artifacts for Later Phases

We persist all fitted objects and preprocessed matrices to ensure seamless hand-off to:
- Phase 4: Model Training
- Phase 5: Evaluation & Error Analysis
- Phase 6: Inference Pipeline & Demo


In [8]:
# 8.1 Persist Fitted Artifacts
vectorizer_path = MODELS_DIR / "tfidf_vectorizer.joblib"
joblib.dump(vectorizer, vectorizer_path)
print(f"Saved fitted TfidfVectorizer to: {vectorizer_path.resolve()}")

# Save complete preprocessed split cache
cache_path = CACHE_DIR / "processed_data_cache.joblib"
cache_payload = {
    "X_train": X_train,
    "y_train": y_train,
    "X_val": X_val,
    "y_val": y_val,
    "X_test": X_test,
    "y_test": y_test,
    "train_indices": df_train.index.values,
    "val_indices": df_val.index.values,
    "test_indices": df_test.index.values,
    "test_raw_texts": df_test['Resume_str'].tolist(),
    "test_clean_texts": df_test['Clean_text'].tolist(),
    "test_categories": df_test['Category'].tolist()
}
joblib.dump(cache_payload, cache_path)
print(f"Saved preprocessed data splits cache to: {cache_path.resolve()}")

# 8.2 Save Feature Pipeline Summary Report
pipeline_summary = {
    "phase": "Phase 3: Text Preprocessing & Feature Engineering",
    "preprocessing": {
        "technical_tokens_protected": list(TECH_TOKEN_REPLACEMENTS.values()),
        "stopwords_filtered": len(DOMAIN_SAFE_STOPWORDS),
        "regex_steps": [
            "Lowercase normalization",
            "Technical term protection",
            "HTML & entity removal",
            "Email & phone redaction",
            "URL stripping",
            "Special characters cleanup",
            "Domain-safe stopword removal",
            "Whitespace collapse"
        ]
    },
    "split_breakdown": {
        "train_samples": int(X_train.shape[0]),
        "val_samples": int(X_val.shape[0]),
        "test_samples": int(X_test.shape[0]),
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
        "test_ratio": TEST_RATIO,
        "random_state": RANDOM_STATE
    },
    "feature_pipeline": {
        "type": "TfidfVectorizer",
        "sublinear_tf": True,
        "ngram_range": [1, 2],
        "max_features": 12000,
        "vocabulary_size": vocab_size,
        "matrix_shape_train": list(X_train.shape),
        "matrix_shape_val": list(X_val.shape),
        "matrix_shape_test": list(X_test.shape),
        "sparsity_percentage": float(round((1.0 - X_train.nnz / (X_train.shape[0] * X_train.shape[1])) * 100, 2))
    },
    "leakage_verification": {
        "fit_on_train_only": True,
        "zero_test_exposure": True,
        "all_24_classes_preserved": True
    },
    "saved_artifacts": [
        str(vectorizer_path),
        str(label_encoder_path),
        str(cache_path)
    ]
}

summary_report_path = REPORTS_DIR / "feature_pipeline_summary.json"
with open(summary_report_path, "w", encoding="utf-8") as f:
    json.dump(pipeline_summary, f, indent=4)
print(f"Saved feature pipeline summary to: {summary_report_path.resolve()}")


Saved fitted TfidfVectorizer to: D:\ResumeForge\artifacts\models\tfidf_vectorizer.joblib
Saved preprocessed data splits cache to: D:\ResumeForge\artifacts\processed_data_cache.joblib
Saved feature pipeline summary to: D:\ResumeForge\artifacts\reports\feature_pipeline_summary.json


In [9]:
# 8.3 Final Phase 3 Summary
print("="*75)
print("PHASE 3: TEXT PREPROCESSING & FEATURE ENGINEERING COMPLETE")
print("="*75)
print(f"• Cleaning Function Ready   : clean_resume_text() validated for training & inference")
print(f"• Technical Terms Preserved : C++, C#, .NET, SQL, AWS, Docker, CI/CD, Power BI, Java, React, ML")
print(f"• Stratified Partitions     : Train = {X_train.shape[0]:,} | Val = {X_val.shape[0]:,} | Test = {X_test.shape[0]:,}")
print(f"• Feature Space             : {vocab_size:,} unigrams & bigrams (sublinear TF scaling)")
print(f"• Zero Data Leakage         : Vectorizer and vocabulary fitted strictly on training data")
print(f"• Saved Artifacts           : tfidf_vectorizer.joblib, label_encoder.joblib, processed_data_cache.joblib")
print(f"• Note                      : Test set evaluation intentionally reserved for Phase 5")
print("="*75)
print("Ready for Phase 4: Classical Machine Learning & Deep Neural Network Training.")


PHASE 3: TEXT PREPROCESSING & FEATURE ENGINEERING COMPLETE
• Cleaning Function Ready   : clean_resume_text() validated for training & inference
• Technical Terms Preserved : C++, C#, .NET, SQL, AWS, Docker, CI/CD, Power BI, Java, React, ML
• Stratified Partitions     : Train = 1,738 | Val = 373 | Test = 373
• Feature Space             : 12,000 unigrams & bigrams (sublinear TF scaling)
• Zero Data Leakage         : Vectorizer and vocabulary fitted strictly on training data
• Saved Artifacts           : tfidf_vectorizer.joblib, label_encoder.joblib, processed_data_cache.joblib
• Note                      : Test set evaluation intentionally reserved for Phase 5
Ready for Phase 4: Classical Machine Learning & Deep Neural Network Training.
